# 故障助手 - 文档处理管线（Kaggle 版）

**用途**：在 Kaggle 的 16GB 内存环境中处理大型 PDF，解决本地内存不足（`std::bad_alloc`）问题。

**处理流程**：Docling 解析 → 文本切块 → 表格摘要（qwen-turbo）→ 图片上传 B2 → VLM 描述（qwen-vl-plus）→ 写入 Qdrant Cloud

---
## 使用前准备

1. 在 Kaggle 左侧边栏 **Add-ons → Secrets** 中添加以下密钥：
   - `DASHSCOPE_API_KEY`
   - `QDRANT_URL`（如 `abc123.us-east-1-0.aws.cloud.qdrant.io`）
   - `QDRANT_API_KEY`
   - `B2_ENDPOINT`（如 `https://s3.us-west-004.backblazeb2.com`）
   - `B2_KEY_ID`
   - `B2_APP_KEY`

2. 在 Kaggle 右侧边栏 **Input → Upload** 上传要处理的 PDF 文件（或添加为 Dataset）。

3. 在最后一个 Cell 中设置 `PDF_PATH` 和 `SOURCE_FILE` 后运行全部 Cell。

In [ ]:
# ── 安装依赖（首次运行约需 5-10 分钟）──
!pip install -q \
    docling \
    pypdfium2 \
    langchain-qdrant \
    langchain-community \
    langchain-openai \
    langchain-text-splitters \
    qdrant-client \
    boto3 \
    dashscope

In [ ]:
# ── 从 Kaggle Secrets 加载配置 ──
import os
from kaggle_secrets import UserSecretsClient

_s = UserSecretsClient()

def _get(key, default=""):
    try:
        return _s.get_secret(key) or default
    except Exception:
        return os.environ.get(key, default)

DASHSCOPE_API_KEY    = _get("DASHSCOPE_API_KEY")
QDRANT_URL           = _get("QDRANT_URL")
QDRANT_API_KEY       = _get("QDRANT_API_KEY")
QDRANT_COLLECTION    = _get("QDRANT_COLLECTION_NAME", "fault_assistant_rag")
B2_ENDPOINT          = _get("B2_ENDPOINT")
B2_KEY_ID            = _get("B2_KEY_ID")
B2_APP_KEY           = _get("B2_APP_KEY")
B2_BUCKET            = _get("B2_BUCKET_NAME", "after-sales-media")
B2_REGION            = _get("B2_REGION", "us-west-004")

# 自动补全 https://
if QDRANT_URL and not QDRANT_URL.startswith(("http://", "https://")):
    QDRANT_URL = "https://" + QDRANT_URL
if B2_ENDPOINT and not B2_ENDPOINT.startswith(("http://", "https://")):
    B2_ENDPOINT = "https://" + B2_ENDPOINT

os.environ["DASHSCOPE_API_KEY"] = DASHSCOPE_API_KEY

EMBEDDING_MODEL     = "text-embedding-v3"
EMBEDDING_DIMENSION = 1024
LLM_MODEL           = "qwen3-max"
VLM_MODEL           = "qwen-vl-max"

print("✅ 配置加载完成")
print(f"  Qdrant : {QDRANT_URL}")
print(f"  B2     : {B2_ENDPOINT}")
print(f"  Bucket : {B2_BUCKET}")

In [ ]:
# ── Docling 文档解析（两遍处理，分离内存峰值）──
import gc
import io
import uuid
from typing import List, Optional

_PDF_CHUNK_SIZE        = 15
_INLINE_TEXT_THRESHOLD = 5


# ── 跨页表格合并辅助函数 ──

def _parse_md_col_count(md: str) -> int:
    """解析 Markdown 表格的列数（取第一行非分隔行）"""
    for line in md.splitlines():
        stripped = line.strip()
        if not (stripped.startswith('|') and stripped.endswith('|')):
            continue
        inner = stripped[1:-1]
        if all(c in '|-: ' for c in inner):
            continue
        return stripped.count('|') - 1
    return 0


def _get_md_data_rows(md: str) -> List[str]:
    """提取 Markdown 表格的数据行（跳过表头行和分隔行）"""
    lines = md.splitlines()
    sep_found = False
    data_rows = []
    for line in lines:
        stripped = line.strip()
        if not sep_found:
            if stripped.startswith('|') and stripped.endswith('|'):
                inner = stripped[1:-1]
                if all(c in '|-: ' for c in inner):
                    sep_found = True
        elif stripped:
            data_rows.append(line)
    return data_rows


def _merge_md_tables(md_list: List[str]) -> str:
    """将多个 Markdown 表格合并为一个（保留第一个的表头，追加后续的数据行）"""
    if not md_list:
        return ""
    if len(md_list) == 1:
        return md_list[0]
    result = md_list[0].rstrip()
    for md in md_list[1:]:
        rows = _get_md_data_rows(md)
        if rows:
            result += "\n" + "\n".join(rows)
    return result


def _build_table_merge_groups(doc):
    """
    分析 doc 中所有 TableItem，识别跨页分割的同一张表格。

    判断条件（三条同时满足）：
    1. 相邻 TableItem 页码差恰好为 1
    2. 两表之间（文档顺序）无有意义的 TextItem
       （忽略：page_header / page_footer、长度 ≤ 6 的纯数字/短横线文本页码、
              以"续表"/"（续）"/"(续)"开头的短标注）
    3. 两表列数相同

    Returns:
        groups:   List[List[TableItem]]，每组属于同一逻辑表格
        md_cache: {id(item): markdown}，供调用方复用，避免重复导出
    """
    from docling.datamodel.document import TextItem, TableItem

    IGNORED_LABELS = {"page_header", "page_footer"}

    all_items = list(doc.iterate_items())
    table_positions = []

    for idx, (item, _) in enumerate(all_items):
        if isinstance(item, TableItem):
            page_no = item.prov[0].page_no if item.prov else None
            table_positions.append((idx, item, page_no))

    if not table_positions:
        return [], {}

    md_cache: dict = {}

    def get_md(item):
        key = id(item)
        if key not in md_cache:
            md_cache[key] = item.export_to_markdown(doc)
        return md_cache[key]

    groups: List[List] = [[table_positions[0][1]]]

    for i in range(1, len(table_positions)):
        prev_idx, prev_item, prev_page = table_positions[i - 1]
        curr_idx, curr_item, curr_page = table_positions[i]

        if prev_page is None or curr_page is None or curr_page != prev_page + 1:
            groups.append([curr_item])
            continue

        has_significant = False
        for j in range(prev_idx + 1, curr_idx):
            mid_item, _ = all_items[j]
            if not isinstance(mid_item, TextItem):
                continue
            label = getattr(mid_item, "label", None)
            if label in IGNORED_LABELS:
                continue
            text = mid_item.text.strip()
            if not text:
                continue
            if len(text) <= 6 and all(c in '0123456789- \t' for c in text):
                continue
            if text.startswith(("续表", "（续）", "(续)")):
                continue
            has_significant = True
            break

        if has_significant:
            groups.append([curr_item])
            continue

        if _parse_md_col_count(get_md(prev_item)) != _parse_md_col_count(get_md(curr_item)):
            groups.append([curr_item])
            continue

        groups[-1].append(curr_item)

    return groups, md_cache


# ── Docling 转换器与工具函数 ──

def _make_converter(extract_images: bool = False, table_structure: bool = True):
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.datamodel.base_models import InputFormat

    opts = PdfPipelineOptions()
    opts.do_ocr = False
    opts.do_table_structure = table_structure
    opts.generate_picture_images = extract_images
    return DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})


def _get_pdf_page_count(file_path: str) -> int:
    import pypdfium2 as pdfium
    pdf = pdfium.PdfDocument(file_path)
    count = len(pdf)
    pdf.close()
    return count


def _separate_elements(doc):
    """提取表格列表（使用跨页合并逻辑）"""
    from docling.datamodel.document import TableItem

    groups, md_cache = _build_table_merge_groups(doc)
    skip_ids: set = set()
    first_md: dict = {}
    for group in groups:
        md_list = [md_cache.get(id(item), item.export_to_markdown(doc)) for item in group]
        first_md[id(group[0])] = _merge_md_tables(md_list)
        for item in group[1:]:
            skip_ids.add(id(item))

    tables = []
    for item, _ in doc.iterate_items():
        if isinstance(item, TableItem):
            if id(item) in skip_ids:
                continue
            tables.append(first_md.get(id(item), item.export_to_markdown(doc)))
    return tables


def _collect_inline_bboxes(doc) -> dict:
    from docling.datamodel.document import PictureItem
    inline_bboxes: dict = {}
    for item, _ in doc.iterate_items():
        if not isinstance(item, PictureItem) or not item.prov:
            continue
        try:
            bbox = item.prov[0].bbox
            pno  = item.prov[0].page_no
            w = abs(bbox.r - bbox.l)
            h = abs(bbox.t - bbox.b)
            if w < 350 and h < 250:
                inline_bboxes.setdefault(pno, []).append(bbox)
        except Exception:
            pass
    return inline_bboxes


def _text_in_inline_bbox(item, inline_bboxes: dict) -> bool:
    if not item.prov:
        return False
    try:
        prov   = item.prov[0]
        bboxes = inline_bboxes.get(prov.page_no)
        if not bboxes:
            return False
        tb = prov.bbox
        cx = (tb.l + tb.r) / 2
        cy = (tb.t + tb.b) / 2
        for ib in bboxes:
            if (min(ib.l, ib.r) <= cx <= max(ib.l, ib.r) and
                    min(ib.t, ib.b) <= cy <= max(ib.t, ib.b)):
                return True
    except Exception:
        pass
    return False


def _analyze_with_images(doc, img_offset: int = 0):
    from docling.datamodel.document import TextItem, TableItem, PictureItem

    inline_bboxes = _collect_inline_bboxes(doc)

    groups, md_cache = _build_table_merge_groups(doc)
    skip_ids: set = set()
    for group in groups:
        for item in group[1:]:
            skip_ids.add(id(item))

    texts: List[str] = []
    images: List[dict] = []
    img_group: List[dict] = []
    img_counter = [img_offset]

    def finalize_group():
        if not img_group:
            return
        gid = str(uuid.uuid4())
        for entry in img_group:
            entry["group_id"] = gid
            images.append(entry)
        img_group.clear()

    for item, _ in doc.iterate_items():
        if isinstance(item, TableItem):
            if id(item) in skip_ids:
                continue
            finalize_group()
        elif isinstance(item, PictureItem):
            pil_img = None
            try:
                pil_img = item.get_image(doc)
            except Exception:
                pass
            if pil_img is None:
                continue
            w, h = pil_img.size
            caption = ""
            try:
                caption = item.caption_text(doc) or ""
            except Exception:
                pass
            page_number = None
            try:
                if item.prov:
                    page_number = item.prov[0].page_no
            except Exception:
                pass
            buf = io.BytesIO()
            pil_img.save(buf, format="PNG")
            img_counter[0] += 1
            if w < 300 and h < 200:
                finalize_group()
                pid = str(uuid.uuid4())
                texts.append(f"[[IMG:{pid}]]")
                img_group.append({
                    "bytes": buf.getvalue(),
                    "filename": f"inline_{img_counter[0]:03d}.png",
                    "caption": caption.strip(),
                    "page_number": page_number,
                    "width": w, "height": h,
                    "group_id": None, "inline": True, "placeholder_id": pid,
                })
                finalize_group()
            else:
                img_group.append({
                    "bytes": buf.getvalue(),
                    "filename": f"image_{img_counter[0]:03d}.png",
                    "caption": caption.strip(),
                    "page_number": page_number,
                    "width": w, "height": h, "group_id": None,
                })
        elif isinstance(item, TextItem):
            if getattr(item, "label", None) in ("page_header", "page_footer"):
                continue
            text = item.text.strip()
            if not text:
                continue
            if _text_in_inline_bbox(item, inline_bboxes):
                continue
            if len(text) >= _INLINE_TEXT_THRESHOLD:
                finalize_group()
            texts.append(text)

    finalize_group()
    return texts, images


def capture_table_images(pdf_path: str, doc) -> List[Optional[bytes]]:
    """
    对 doc 中的每张表格，用 pypdfium2 裁剪 PDF 页面区域渲染为 PNG。
    跨页分割的同一张表格（由 _build_table_merge_groups 识别）竖向拼接为一张图。

    坐标系自动识别：
    - bbox.t >= bbox.b → BOTTOMLEFT，y_pixel = (page_h - bbox_y) * scale
    - bbox.t <  bbox.b → TOPLEFT，  y_pixel = bbox_y * scale
    """
    from PIL import Image as PILImage
    import pypdfium2 as pdfium

    def _crop_prov(pdf, prov, scale: float):
        try:
            page_h  = pdf[prov.page_no - 1].get_height()
            bbox    = prov.bbox
            pil_img = pdf[prov.page_no - 1].render(scale=scale).to_pil()

            if bbox.t >= bbox.b:        # BOTTOMLEFT
                raw_y0 = (page_h - bbox.t) * scale
                raw_y1 = (page_h - bbox.b) * scale
            else:                        # TOPLEFT
                raw_y0 = bbox.t * scale
                raw_y1 = bbox.b * scale

            pad = 8
            x0 = max(0,              int(bbox.l * scale) - pad)
            y0 = max(0,              int(raw_y0) - pad)
            x1 = min(pil_img.width,  int(bbox.r * scale) + pad)
            y1 = min(pil_img.height, int(raw_y1) + pad)

            if x0 >= x1 or y0 >= y1:
                return None
            return pil_img.crop((x0, y0, x1, y1))
        except Exception as e:
            print(f"⚠️ prov 裁剪失败 (页{prov.page_no}): {e}")
            return None

    groups, _ = _build_table_merge_groups(doc)

    result: List[Optional[bytes]] = []
    try:
        pdf = pdfium.PdfDocument(pdf_path)
    except Exception as e:
        print(f"⚠️ 打开 PDF 失败，跳过表格图片捕获: {e}")
        return [None] * len(groups)

    scale = 2.0
    try:
        for group in groups:
            all_provs = []
            for item in group:
                all_provs.extend(item.prov or [])

            if not all_provs:
                result.append(None)
                continue

            try:
                # 每页只保留面积最大的 prov（去除同页重复/ML误合并）
                page_best: dict = {}
                for p in all_provs:
                    area = abs((p.bbox.r - p.bbox.l) * (p.bbox.t - p.bbox.b))
                    if p.page_no not in page_best or area > page_best[p.page_no][1]:
                        page_best[p.page_no] = (p, area)

                sorted_pages = sorted(page_best.keys())

                # 非连续页码 → 只取面积最大的单页
                if len(sorted_pages) > 1:
                    all_consecutive = all(
                        sorted_pages[i + 1] == sorted_pages[i] + 1
                        for i in range(len(sorted_pages) - 1)
                    )
                    if not all_consecutive:
                        best_pno = max(page_best, key=lambda k: page_best[k][1])
                        sorted_pages = [best_pno]

                # 裁剪每页区域（含坐标系自动识别），连续多页竖向拼接
                parts = []
                for pno in sorted_pages:
                    crop = _crop_prov(pdf, page_best[pno][0], scale)
                    if crop is not None:
                        parts.append(crop)

                if not parts:
                    result.append(None)
                    continue

                if len(parts) == 1:
                    final_img = parts[0]
                else:
                    gap = 6
                    total_h = sum(p.height for p in parts) + gap * (len(parts) - 1)
                    max_w   = max(p.width  for p in parts)
                    final_img = PILImage.new("RGB", (max_w, total_h), (255, 255, 255))
                    y_off = 0
                    for part in parts:
                        final_img.paste(part, (0, y_off))
                        y_off += part.height + gap

                buf = io.BytesIO()
                final_img.save(buf, format="PNG")
                result.append(buf.getvalue())
            except Exception as e:
                print(f"⚠️ 表格图片处理失败: {e}")
                result.append(None)
    finally:
        pdf.close()

    return result


def parse_pdf(file_path: str):
    total = _get_pdf_page_count(file_path)
    print(f"📄 共 {total} 页，按每批 {_PDF_CHUNK_SIZE} 页处理")

    all_texts, all_images = [], []
    conv_img = _make_converter(extract_images=True, table_structure=False)
    for start in range(1, total + 1, _PDF_CHUNK_SIZE):
        end = min(start + _PDF_CHUNK_SIZE - 1, total)
        print(f"  第一遍（文本/图片）第 {start}-{end} 页...", end=" ")
        try:
            result = conv_img.convert(file_path, page_range=(start, end))
            t, imgs = _analyze_with_images(result.document, img_offset=len(all_images))
            all_texts.extend(t)
            all_images.extend(imgs)
            print(f"文本 {len(t)} 段，图片 {len(imgs)} 张")
        except Exception as e:
            print(f"❌ 失败: {e}")
        finally:
            del result
            gc.collect()

    all_tables, all_table_images = [], []
    conv_text = _make_converter(extract_images=False, table_structure=True)
    for start in range(1, total + 1, _PDF_CHUNK_SIZE):
        end = min(start + _PDF_CHUNK_SIZE - 1, total)
        print(f"  第二遍（表格）第 {start}-{end} 页...", end=" ")
        try:
            result = conv_text.convert(file_path, page_range=(start, end))
            tbs   = _separate_elements(result.document)
            timgs = capture_table_images(file_path, result.document)
            all_tables.extend(tbs)
            all_table_images.extend(timgs)
            print(f"表格 {len(tbs)} 个（图片 {sum(1 for x in timgs if x)} 张成功）")
        except Exception as e:
            print(f"❌ 失败: {e}")
        finally:
            del result
            gc.collect()

    print(f"\n✅ 解析完成：文本 {len(all_texts)} 段 / 表格 {len(all_tables)} 个 / 图片 {len(all_images)} 张")
    return all_texts, all_tables, all_images, all_table_images

In [ ]:
# ── 文本切块 ──
from langchain_text_splitters import RecursiveCharacterTextSplitter

_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "！", "？", ".", " "],
)

def split_texts(texts: List[str]) -> List[str]:
    combined = "\n\n".join(t for t in texts if t.strip())
    return _splitter.split_text(combined)

print("✅ 文本切块模块就绪")

In [ ]:
# ── LLM 服务：表格摘要 + 图片描述（qwen-vl-plus）──
import base64
from langchain_openai import ChatOpenAI

_DASHSCOPE_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"


def summarize_table(table_markdown: str) -> str:
    """用 qwen-turbo 为 Markdown 表格生成自然语言摘要"""
    llm = ChatOpenAI(
        model=LLM_MODEL,
        api_key=DASHSCOPE_API_KEY,
        base_url=_DASHSCOPE_BASE,
        temperature=0,
    )
    prompt = (
        "请用中文简洁概括以下表格的内容，包括：\n"
        "1. 这个表格是关于什么的\n"
        "2. 包含哪些关键字段/列\n"
        "3. 关键数据要点\n\n"
        f"表格内容：\n{table_markdown}"
    )
    try:
        return llm.invoke(prompt).content.strip()
    except Exception as e:
        return f"表格摘要生成失败: {e}"


def describe_image(image_bytes: bytes, caption: str = "") -> str:
    """用 qwen-vl-plus 为图片生成中文描述"""
    b64 = base64.b64encode(image_bytes).decode("utf-8")
    ctx = caption or "无"
    prompt = (
        "请用中文详细描述这张图片的内容，这是一个售后客服知识库中的图片。\n\n"
        "描述要求：\n"
        "1. 描述图片中显示的产品、零件、操作流程或故障现象\n"
        "2. 如果是产品图，指出外观特征、标注的名称或型号\n"
        "3. 如果是流程图/示意图，按步骤描述流程\n"
        "4. 如果是故障/问题图片，描述故障现象和可能位置\n"
        "5. 描述要便于通过文字搜索找到这张图片\n\n"
        f"图片在文档中的上下文说明：{ctx}"
    )
    llm = ChatOpenAI(
        model=VLM_MODEL,
        api_key=DASHSCOPE_API_KEY,
        base_url=_DASHSCOPE_BASE,
        temperature=0,
    )
    messages = [{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            {"type": "text", "text": prompt},
        ],
    }]
    try:
        return llm.invoke(messages).content.strip()
    except Exception as e:
        fallback = caption.strip()
        return f"图片描述生成失败: {e}" + (f"（标题：{fallback}）" if fallback else "")


print("✅ LLM 服务模块就绪")

In [ ]:
# ── B2 对象存储（S3 兼容 API）──
import boto3
from botocore.config import Config
from pathlib import Path

_PRESIGNED_EXPIRES = 7 * 24 * 3600
_CONTENT_TYPE_MAP = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg"}


class B2Client:
    def __init__(self):
        self.bucket = B2_BUCKET
        self.s3 = boto3.client(
            "s3",
            endpoint_url=B2_ENDPOINT,
            aws_access_key_id=B2_KEY_ID,
            aws_secret_access_key=B2_APP_KEY,
            config=Config(region_name=B2_REGION, signature_version="s3v4"),
        )

    def upload_bytes(self, data: bytes, filename: str) -> dict:
        ext = Path(filename).suffix.lower() or ".png"
        object_key = f"images/{uuid.uuid4().hex}{ext}"
        content_type = _CONTENT_TYPE_MAP.get(ext, "image/png")
        self.s3.upload_fileobj(
            io.BytesIO(data), self.bucket, object_key,
            ExtraArgs={"ContentType": content_type},
        )
        url = self.s3.generate_presigned_url(
            "get_object",
            Params={"Bucket": self.bucket, "Key": object_key},
            ExpiresIn=_PRESIGNED_EXPIRES,
        )
        return {"object_key": object_key, "url": url}


b2 = B2Client()
print("✅ B2 客户端就绪")

In [ ]:
# ── Qdrant + 嵌入模型初始化 ──
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue, FilterSelector
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embeddings = DashScopeEmbeddings(model=EMBEDDING_MODEL)

# 确保 collection 存在
existing = [c.name for c in qdrant.get_collections().collections]
if QDRANT_COLLECTION not in existing:
    qdrant.create_collection(
        collection_name=QDRANT_COLLECTION,
        vectors_config=VectorParams(size=EMBEDDING_DIMENSION, distance=Distance.COSINE),
    )
    print(f"✅ 已创建 collection: {QDRANT_COLLECTION}")
else:
    count = qdrant.count(collection_name=QDRANT_COLLECTION).count
    print(f"✅ collection 已存在，当前 {count} 条向量")

# 创建 payload 索引
for field in (
    "metadata.source",
    "metadata.type",
    "metadata.doc_id",
    "metadata.group_id",
    "metadata.version",
    "metadata.doc_type",
):
    try:
        qdrant.create_payload_index(
            collection_name=QDRANT_COLLECTION, field_name=field, field_schema="keyword"
        )
    except Exception:
        pass

vectorstore = QdrantVectorStore(
    client=qdrant, collection_name=QDRANT_COLLECTION, embedding=embeddings
)
print("✅ Qdrant 向量存储就绪")

In [ ]:
# ── 删除旧数据（同名文件覆盖）──
def delete_by_source(source_file: str):
    offset = None
    doc_ids = []
    while True:
        points, next_offset = qdrant.scroll(
            collection_name=QDRANT_COLLECTION,
            scroll_filter=Filter(must=[
                FieldCondition(key="metadata.source", match=MatchValue(value=source_file))
            ]),
            limit=100, offset=offset, with_payload=True, with_vectors=False,
        )
        for p in points:
            doc_id = p.payload.get("metadata", {}).get("doc_id")
            if doc_id:
                doc_ids.append(doc_id)
        if next_offset is None:
            break
        offset = next_offset

    if doc_ids:
        qdrant.delete(
            collection_name=QDRANT_COLLECTION,
            points_selector=FilterSelector(
                filter=Filter(must=[
                    FieldCondition(key="metadata.source", match=MatchValue(value=source_file))
                ])
            ),
        )
        print(f"🗑️ 已删除来源 '{source_file}' 的 {len(doc_ids)} 条旧向量")


# ── 写入 Qdrant ──
def index_texts(chunks, source_file: str, version: str = "", doc_type: str = ""):
    if not chunks:
        return
    docs = [
        Document(
            page_content=chunk,
            metadata={
                "doc_id": str(uuid.uuid4()),
                "original_content": chunk,
                "source": source_file,
                "type": "text_chunk",
                "version": version,
                "doc_type": doc_type,
            },
        )
        for chunk in chunks
    ]
    vectorstore.add_documents(docs)
    print(f"  📝 写入 {len(docs)} 个文本块")


def index_tables(tables: list, summaries: list, source_file: str,
                 table_image_urls: list = None,
                 table_image_object_keys: list = None,
                 version: str = "",
                 doc_type: str = ""):
    """
    写入表格向量。
    table_image_urls:        与 tables 等长，元素为 B2 预签名 URL 或 None。
    table_image_object_keys: 与 tables 等长，元素为 B2 object_key 或 None（用于刷新过期 URL）。
    version:                 版本号，如 "1101"。
    doc_type:                文档类型，如 "user_manual" / "test_report" / "product_spec"。
    """
    if not tables or not summaries:
        return
    docs = [
        Document(
            page_content=summary,
            metadata={
                "doc_id": str(uuid.uuid4()),
                "original_content": tables[i],
                "source": source_file,
                "type": "table",
                "table_image_url": (
                    table_image_urls[i]
                    if table_image_urls and i < len(table_image_urls)
                    else None
                ) or "",
                "table_image_object_key": (
                    table_image_object_keys[i]
                    if table_image_object_keys and i < len(table_image_object_keys)
                    else None
                ) or "",
                "version": version,
                "doc_type": doc_type,
            },
        )
        for i, summary in enumerate(summaries)
    ]
    vectorstore.add_documents(docs)
    has_img = sum(1 for d in docs if d.metadata.get("table_image_url"))
    print(f"  📊 写入 {len(docs)} 个表格（含图片 {has_img} 个）")


def index_image(img_info: dict, source_file: str, version: str = "", doc_type: str = ""):
    upload = b2.upload_bytes(img_info["bytes"], img_info["filename"])
    description = describe_image(img_info["bytes"], caption=img_info.get("caption", ""))
    doc_id = str(uuid.uuid4())
    group_id = img_info.get("group_id")
    media_payload = json.dumps({
        "type": "image", "url": upload["url"], "description": description,
        "filename": img_info["filename"], "group_id": group_id,
    }, ensure_ascii=False)
    vectorstore.add_documents([
        Document(
            page_content=description,
            metadata={
                "doc_id": doc_id,
                "original_content": media_payload,
                "media_url": upload["url"],
                "object_key": upload["object_key"],
                "source": source_file,
                "type": "image",
                "page_number": img_info.get("page_number"),
                "group_id": group_id,
                "version": version,
                "doc_type": doc_type,
            },
        )
    ])
    return doc_id


print("✅ 索引函数就绪")

In [ ]:
# ── 主管线 ──

def _replace_inline_placeholders(text: str, mapping: dict) -> str:
    for pid, replacement in mapping.items():
        text = text.replace(f"[[IMG:{pid}]]", replacement)
    return text


def process_pdf(pdf_path: str, source_file: str = None, version: str = "", doc_type: str = ""):
    """
    完整处理一个 PDF 文件并写入 Qdrant + B2

    Args:
        pdf_path:    PDF 文件在 Kaggle 中的路径
        source_file: 写入 Qdrant 的 source 标识（默认用文件名）
        version:     版本号，如 "1101"（留空则不关联版本）
        doc_type:    文档业务类型：user_manual / test_report / product_spec（留空则不标注）
    """
    from pathlib import Path
    if source_file is None:
        source_file = Path(pdf_path).name

    print(f"\n{'='*60}")
    print(f"📂 处理文件：{source_file}")
    if version:
        print(f"   版本号：{version}  文档类型：{doc_type or '未指定'}")
    print(f"{'='*60}")

    # 1. 解析
    print("\n[1/5] 解析文档...")
    texts, tables, images, table_images = parse_pdf(pdf_path)
    inline_imgs  = [img for img in images if img.get("inline")]
    regular_imgs = [img for img in images if not img.get("inline")]
    print(f"  内联图片 {len(inline_imgs)} 张 / 独立图片 {len(regular_imgs)} 张 / 表格图片 {sum(1 for x in table_images if x)} 张")

    # 2. 内联图片：上传 B2 → 建立占位符替换表
    placeholder_map = {}
    if inline_imgs:
        print(f"\n[2/5] 上传 {len(inline_imgs)} 张内联图片到 B2...")
        for img in inline_imgs:
            try:
                upload = b2.upload_bytes(img["bytes"], img["filename"])
                alt = img.get("caption") or "图示"
                placeholder_map[img["placeholder_id"]] = f"![{alt}]({upload['url']})"
                print(f"  ✅ {img['filename']}")
            except Exception as e:
                print(f"  ⚠️ {img['filename']} 上传失败: {e}")
                placeholder_map[img["placeholder_id"]] = f"[{img.get('caption') or '图示'}]"
    else:
        print("\n[2/5] 无内联图片，跳过")

    # 3. 表格图片：上传 B2 → 同时收集 URL 和 object_key
    table_image_urls = []
    table_image_object_keys = []
    if table_images:
        print(f"\n[3/5] 上传表格图片到 B2...")
        for i, img_bytes in enumerate(table_images):
            if img_bytes:
                try:
                    upload = b2.upload_bytes(img_bytes, f"table_{i+1:03d}.png")
                    table_image_urls.append(upload["url"])
                    table_image_object_keys.append(upload["object_key"])
                    print(f"  ✅ table_{i+1:03d}.png")
                except Exception as e:
                    print(f"  ⚠️ table_{i+1:03d}.png 上传失败: {e}")
                    table_image_urls.append(None)
                    table_image_object_keys.append(None)
            else:
                table_image_urls.append(None)
                table_image_object_keys.append(None)
    else:
        print("\n[3/5] 无表格图片，跳过")
        table_image_urls = [None] * len(tables)
        table_image_object_keys = [None] * len(tables)

    # 4. 文本切块（替换占位符后）+ 表格摘要 → 写入 Qdrant
    print("\n[4/5] 文本切块 + 写入 Qdrant...")
    chunks = split_texts(texts)
    if placeholder_map:
        chunks = [_replace_inline_placeholders(c, placeholder_map) for c in chunks]
    print(f"  切块完成：{len(texts)} 段 → {len(chunks)} 块")

    delete_by_source(source_file)
    index_texts(chunks, source_file, version=version, doc_type=doc_type)

    if tables:
        print(f"  生成 {len(tables)} 个表格摘要...")
        summaries = []
        for i, tbl in enumerate(tables):
            print(f"    表格 {i+1}/{len(tables)}...", end=" ")
            s = summarize_table(tbl)
            summaries.append(s)
            print("完成")
        index_tables(
            tables, summaries, source_file,
            table_image_urls=table_image_urls,
            table_image_object_keys=table_image_object_keys,
            version=version,
            doc_type=doc_type,
        )

    # 5. 独立图片：上传 B2 + VLM 描述 → 写入 Qdrant
    if regular_imgs:
        print(f"\n[5/5] 处理 {len(regular_imgs)} 张独立图片（上传 B2 + VLM 描述）...")
        ok, fail = 0, 0
        for i, img in enumerate(regular_imgs):
            print(f"  图片 {i+1}/{len(regular_imgs)} [{img['filename']}]...", end=" ")
            try:
                index_image(img, source_file, version=version, doc_type=doc_type)
                ok += 1
                print("✅")
            except Exception as e:
                fail += 1
                print(f"❌ {e}")
        print(f"  图片处理完成：成功 {ok} / 失败 {fail}")
    else:
        print("\n[5/5] 无独立图片，跳过")

    total = qdrant.count(collection_name=QDRANT_COLLECTION).count
    print(f"\n✅ 全部完成！Qdrant collection 现有 {total} 条向量")


print("✅ 主管线就绪")

In [ ]:
# ── 在此处设置 PDF 路径后运行 ──
#
# 上传方法：
#   Kaggle 右侧边栏 → Input → Upload → 上传 PDF
#   上传后路径格式为 /kaggle/input/<dataset-slug>/<filename>.pdf
#
# VERSION  填写版本号，如 "1101"（基础版）、"1102"（特殊版）
# DOC_TYPE 填写文档类型：
#   "user_manual"   → 用户手册（基础版本通用功能）
#   "test_report"   → 测试报告（各版本新增差异内容）
#   "product_spec"  → 产品说明书

PDF_PATH    = "/kaggle/input/my-dataset/your_file.pdf"  # ← 修改这里
SOURCE_FILE = None      # None 表示自动使用文件名
VERSION     = "1101"    # ← 填写版本号，留空 "" 表示不关联版本
DOC_TYPE    = "user_manual"  # ← 填写文档类型

process_pdf(PDF_PATH, SOURCE_FILE, version=VERSION, doc_type=DOC_TYPE)